In [ ]:
%uv pip install datasets transformers torch numpy pandas

Using Python 3.12.6 environment at: /usr/local
Audited 5 packages in 27ms
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from datasets import load_dataset, concatenate_datasets
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np
import json
from sklearn.feature_extraction.text import CountVectorizer
from tqdm import tqdm
import os
from huggingface_hub import login

# HF_TOKEN = os.environ["HF_TOKEN"]
# login(token=HF_TOKEN)

In [2]:
model_name="Qwen/Qwen3-4B-Base"

tokenizer=AutoTokenizer.from_pretrained(model_name,trust_remote_code=True, device_map='auto')
model=AutoModelForCausalLM.from_pretrained(model_name,trust_remote_code=True, device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [ ]:
dataset=load_dataset("databricks/databricks-dolly-15k",split="train")
dataset = dataset.map(lambda x: {"length": len(x["response"])})
l_dataset=dataset.filter(lambda x: x["length"] > 1000)
categories=['brainstorming', "closed_qa", 'summarization',"creative_writing", 'information_extraction', "general_qa", "classification", 'open_qa']
selected_subsets=[]
for category in categories:
    subset = l_dataset.filter(lambda x: x["category"] == category)
    sampled_subset=subset.select(range(min(15, len(subset))))
    selected_subsets.append(sampled_subset)
data=concatenate_datasets(selected_subsets)
print(len(data))

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15011 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/1027 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1027 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1027 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1027 [00:00<?, ? examples/s]

120


In [ ]:
system_prompt = """
You are a helpful assistant.

The user will provide input in the following format:
Instruction: The task or question you need to answer.
Context: Additional information that may be relevant to the instruction.

The "instruction" field contains the task or question.
If the instruction requires additional context, it is provided in the context field. You must answer the instruction based on the context if context is available.
If context is not required, the context field will be set to None. In that case, you must follow the instruction accurately based on your knowledge.

Keep your response relevant, clear, and complete.
"""

In [9]:
def make_prompt(instruction, context=None):
    if context=="":
      prompt=f"""Instruction: {instruction}\n"""
      return prompt
    prompt = f"""Instruction: {instruction}\nContext: {context}"""

    return prompt

In [ ]:
def get_response(model, tokenizer, prompt, max_new_tokens=100):
  messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
  # print(messages)
  text=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,enable_thinking=False)
  inputs=tokenizer([text], return_tensors="pt").to(model.device)
  generated_ids=model.generate(**inputs,max_new_tokens=max_new_tokens, eos_token_id=tokenizer.eos_token_id,pad_token_id=tokenizer.eos_token_id)
  output_ids=generated_ids[0][len(inputs.input_ids[0]):].tolist()
  response=tokenizer.decode(output_ids, skip_special_tokens=True)
  return response

In [5]:
def get_base_response(model, tokenizer, prompt, max_new_tokens=100):
  # messages = [
  #       {"role": "system", "content": system_prompt},
  #       {"role": "user", "content": prompt},
  #   ]
  # print(messages)
  #text=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,enable_thinking=False)
  inputs=tokenizer(prompt, return_tensors="pt", enable_thinking=False).to(model.device)
  generated_ids=model.generate(**inputs,max_new_tokens=max_new_tokens, eos_token_id=tokenizer.eos_token_id,pad_token_id=tokenizer.eos_token_id)
  output_ids=generated_ids[0][len(inputs.input_ids[0]):].tolist()
  response=tokenizer.decode(output_ids, skip_special_tokens=True)
  return response

In [11]:
responses=[]
for i in tqdm(range(len(data))):
  prompt=make_prompt(data['instruction'][i],data['context'][i])
  responses.append(get_base_response(model, tokenizer, prompt, max_new_tokens=500))
with open("model_responses_500_tokens_base.json", "w", encoding="utf-8") as f:
    json.dump(responses, f, indent=2, ensure_ascii=False)

100%|██████████| 120/120 [53:15<00:00, 26.63s/it]


In [ ]:
model_name="Qwen/Qwen3-4B-Instruct-2507"

tokenizer=AutoTokenizer.from_pretrained(model_name,trust_remote_code=True, device_map='auto')
model=AutoModelForCausalLM.from_pretrained(model_name,trust_remote_code=True, device_map="auto")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
responses=[]
for i in tqdm(range(len(data))):
  prompt=make_prompt(data['instruction'][i],data['context'][i])
  # print(prompt)
  responses.append(get_response(model, tokenizer, prompt, max_new_tokens=500))
with open("model_responses_500_tokens_instruct.json", "w", encoding="utf-8") as f:
    json.dump(responses, f, indent=2, ensure_ascii=False)

100%|████████████████████████████████████████████████████████████████████████| 120/120 [22:28<00:00, 11.24s/it]


In [ ]:
import re

def extract_thinking_trace(text):

    # Case 1: Has both <think> and </think>
    match = re.search(r"<think>\s*(.*?)\s*</think>", text, flags=re.DOTALL)

    if match:
        thinking_trace = match.group(1).strip()

        # Remove the full think block from the response
        response = re.sub(
            r"<think>\s*.*?\s*</think>",
            "",
            text,
            flags=re.DOTALL
        ).strip()

        return response, thinking_trace

    # Case 2: Has only closing </think>
    if "</think>" in text:
        thinking_trace, response = text.split("</think>", 1)
        return response.strip(), thinking_trace.strip()

    # Case 3: No closing </think>
    # Assume max_new_tokens ran out before thinking completed
    return "", text.strip()

In [ ]:
def get_thinking_response(model, tokenizer, prompt, max_new_tokens=100):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,   # use True for reasoning model
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    output_ids = generated_ids[0][len(inputs.input_ids[0]):].tolist()

    full_output = tokenizer.decode(
        output_ids,
        skip_special_tokens=True
    )

    response, thinking_trace = extract_thinking_trace(full_output)

    return response, thinking_trace

In [ ]:
model_name="Qwen/Qwen3-4B-Thinking-2507"

tokenizer=AutoTokenizer.from_pretrained(model_name,trust_remote_code=True, device_map='auto')
model=AutoModelForCausalLM.from_pretrained(model_name,trust_remote_code=True, device_map="auto")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
data[15]

{'instruction': 'Can you tell me something about Stanley J. Goldberg',
 'context': 'Stanley J. Goldberg (born Maryland, 1939) is a special trial judge of the United States Tax Court.\n\nGoldberg attended public schools in Baltimore, MD. He earned a B.S. from the University of Maryland, School of Business and Public Administration in 1960 and an LL.B. from the University of Maryland School of Law in 1964. He did graduate work in Federal Income Taxation at New York University. Goldberg was admitted to practice in Maryland and New Jersey, 1964 and 1967, and Federal District Court. He began his career as a Tax Attorney in January 1965 with the United States Department of Treasury, Office of Chief Counsel, Internal Revenue Service, in New York City and was initially assigned to the General Litigation function. In 1967, he was reassigned to the Tax Litigation function. In 1976, he was promoted to Special Trial Attorney, and then to Assistant District Counsel in 1984. He was appointed a Speci

In [ ]:
make_prompt(data[15]['instruction'], data[15]['context'])

'Instruction: Can you tell me something about Stanley J. Goldberg\nContext: Stanley J. Goldberg (born Maryland, 1939) is a special trial judge of the United States Tax Court.\n\nGoldberg attended public schools in Baltimore, MD. He earned a B.S. from the University of Maryland, School of Business and Public Administration in 1960 and an LL.B. from the University of Maryland School of Law in 1964. He did graduate work in Federal Income Taxation at New York University. Goldberg was admitted to practice in Maryland and New Jersey, 1964 and 1967, and Federal District Court. He began his career as a Tax Attorney in January 1965 with the United States Department of Treasury, Office of Chief Counsel, Internal Revenue Service, in New York City and was initially assigned to the General Litigation function. In 1967, he was reassigned to the Tax Litigation function. In 1976, he was promoted to Special Trial Attorney, and then to Assistant District Counsel in 1984. He was appointed a Special Trial

In [ ]:
make_prompt(data['instruction'][15], data['context'][15])

'Instruction: Can you tell me something about Stanley J. Goldberg\nContext: Stanley J. Goldberg (born Maryland, 1939) is a special trial judge of the United States Tax Court.\n\nGoldberg attended public schools in Baltimore, MD. He earned a B.S. from the University of Maryland, School of Business and Public Administration in 1960 and an LL.B. from the University of Maryland School of Law in 1964. He did graduate work in Federal Income Taxation at New York University. Goldberg was admitted to practice in Maryland and New Jersey, 1964 and 1967, and Federal District Court. He began his career as a Tax Attorney in January 1965 with the United States Department of Treasury, Office of Chief Counsel, Internal Revenue Service, in New York City and was initially assigned to the General Litigation function. In 1967, he was reassigned to the Tax Litigation function. In 1976, he was promoted to Special Trial Attorney, and then to Assistant District Counsel in 1984. He was appointed a Special Trial

In [ ]:
responses=[]
for i in tqdm(range(len(data))):
  prompt=make_prompt(data['instruction'][i],data['context'][i])
  response,thinking_trace=get_thinking_response(model, tokenizer, prompt, max_new_tokens=1000)
  responses.append((response,thinking_trace))
with open("model_responses_1000_tokens_reasoning.json", "w", encoding="utf-8") as f:
    json.dump(responses, f, indent=2, ensure_ascii=False)

 85%|█████████████████████████████████████████████████████████████▏          | 102/120 [59:55<11:41, 38.95s/it]